In [4]:
import torch
import torch.nn as nn 
import numpy as np
from torch.utils.data import DataLoader, Dataset 
from sklearn.model_selection import train_test_split
import pandas as pd
import csv

In [5]:
def readfiles(datafile, labelfile, sourcefile): 
    '''data reader and simplifier for files that haven't been passed through the algorithm'''
    
    with open(datafile, 'r') as file: 
        reader = csv.reader(file)
        sequences = [
            np.array([list(map(float, item.strip(" []").split())) for item in row])
            for row in reader
        ]
    
    labelread = pd.read_csv(labelfile)
    labels = np.array(labelread['labels'])

    columns = ['x[px]', 'y[px]', 't[s]']
    sourceread = pd.read_csv(sourcefile) 
    sources = np.array(sourceread[columns])

    return sequences, labels, sources

def labelmaker(events=None, sp_density=None, t_density=None, noise=None, filename = None, folder = None): 
    '''creates labels based on my naming convention for different files, keeps it consistent and easy'''
    if folder: 
        folder = folder + '/'

    if filename: 
        datafile = str(filename) 
    else: 
        datafile = str(events) + 'ev_' + str(sp_density) + 'spd_' + str(t_density) + 'td_n' + str(noise)
    

    labelfile = 'labels_' + datafile + '.csv'
    sourcefile = 'sources_' + datafile + '.csv'
    ai_labelfile = datafile + '_results' + '.csv'
    centroidfile = datafile + '_centroids' + '.csv'
    datafile = datafile + '.csv'

    if folder: 
        datafile = folder + datafile 
        centroidfile = folder + centroidfile 
        ai_labelfile = folder + ai_labelfile 
        labelfile = folder + labelfile 
        sourcefile = folder + sourcefile 

    
    return datafile, labelfile, sourcefile, ai_labelfile, centroidfile


In [25]:
filename = '10e_n0'

datafile, labelfile, sourcefile, ai_labelfile, centroidfile = labelmaker(filename = filename)
sequences, labels, sources = readfiles(datafile, labelfile, sourcefile)

short_labels = []
for i in labels: 
    if i not in short_labels: 
        short_labels.append(i)

print(short_labels)

[357, 7270, 1128, 8606, 7846, 1863, 9961, 7610, 3768, 8703]


In [26]:
max_sequence_len =max(len(seq) for seq in sequences)
num_features = 3

## padding the sequences with zeroes
padded_sequences = np.zeros((len(sequences), max_sequence_len, num_features))
for i, seq in enumerate(sequences): 
    padded_sequences[i, :len(seq), :] = seq # copy the real data, leave the padding as 0s

# converting to tensors 
x = torch.tensor(padded_sequences, dtype = torch.float32)
y = torch.tensor(short_labels, dtype = torch.long) # just one label per sequence, can fix this in the sim later 

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.2, random_state=42)

In [ ]:
## Define the dataset and dataloader 

Below is just what chatgpt says to do for LSTM in case i lose it 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split

# --------------------------
# 1. Generate Example Data (Replace with real data)
# --------------------------

# Example photon sequences (time, energy, x, y, z, detector_id)
events = [
    [[0.01, 0.5, 2.1, 3.4, 1.2, 1], [0.02, 0.6, 2.2, 3.5, 1.3, 1]],  # Event A
    [[0.03, 0.4, 1.0, 2.2, 0.8, 3]],  # Event B
    [[0.05, 0.7, 3.0, 4.2, 1.5, 2], [0.06, 0.8, 3.1, 4.3, 1.6, 2]]  # Event C
]

labels = ["A", "B", "C"]  # Neutron event labels

# Encode labels into numerical values
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

# Pad sequences to make them uniform length
max_seq_length = max(len(seq) for seq in events)
num_features = len(events[0][0])  # Number of features per photon

padded_events = np.zeros((len(events), max_seq_length, num_features))
for i, seq in enumerate(events):
    padded_events[i, :len(seq), :] = seq  # Copy real data, leave padding as 0s

# Convert to PyTorch tensors
X = torch.tensor(padded_events, dtype=torch.float32)
y = torch.tensor(encoded_labels, dtype=torch.long)

# Split data into train & test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --------------------------
# 2. Define Dataset and DataLoader
# --------------------------
class PhotonDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = PhotonDataset(X_train, y_train)
test_dataset = PhotonDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

# --------------------------
# 3. Define LSTM Model
# --------------------------
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
        self.hidden_size = hidden_size
        self.num_layers = num_layers

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)  # Initial hidden state
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)  # Initial cell state

        out, _ = self.lstm(x, (h0, c0))  # LSTM forward pass
        out = out[:, -1, :]  # Get last time-step output
        out = self.fc(out)  # Fully connected layer
        return out

# --------------------------
# 4. Training the Model
# --------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define model, loss, optimizer
input_size = num_features  # 6 features per photon
hidden_size = 64
num_layers = 2
num_classes = len(set(encoded_labels))

model = LSTMClassifier(input_size, hidden_size, num_layers, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss / len(train_loader):.4f}")

# --------------------------
# 5. Evaluating the Model
# --------------------------
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

# --------------------------
# 6. Predict on New Photon Sequences
# --------------------------
def predict_event(photon_sequence):
    """Predict the neutron event ID for a new photon sequence"""
    model.eval()
    photon_sequence = torch.tensor(photon_sequence, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(photon_sequence)
        _, predicted = torch.max(output, 1)
    return label_encoder.inverse_transform([predicted.item()])[0]

# Example prediction
new_photon_sequence = [[0.04, 0.6, 2.5, 3.8, 1.4, 2], [0.05, 0.7, 2.6, 3.9, 1.5, 2]]  # Example new event
new_photon_sequence = np.array(new_photon_sequence)
new_photon_sequence = np.pad(new_photon_sequence, ((0, max_seq_length - len(new_photon_sequence)), (0, 0)), mode='constant')

predicted_event = predict_event(new_photon_sequence)
print(f"Predicted Neutron Event: {predicted_event}")
